CFTC Positioning
- Main DataFrame

In [47]:
import glob
import pandas as pd


# Optional: Helper function to keep column names clean and standardized
def clean_columns(df):
  df.columns = (
      df.columns.str.strip()
      .str.lower()
      .str.replace('__', '_', regex=False)
      .str.replace('_othr', '_other', regex=False)
  )
  return df


# 1. Find all your text/CSV files
file_pattern = '../data/raw/cftc/historicals/*.txt'
file_list = glob.glob(file_pattern)

dfs = []
print(f'Reading and combining {len(file_list)} files...\n')

# 2. Loop through each file, clean columns, and add to list
for file in file_list:
  df = pd.read_csv(file, low_memory=False)
  df = clean_columns(df)
  dfs.append(df)
  print(f'✓ Loaded: {file}')

# 3. Concatenate all files into one master dataframe
master_df = pd.concat(dfs, ignore_index=True)

print('\n🎉 Success! All historical text files have been combined.')
print(f'Master Dataframe Shape: {master_df.shape}')
print(f'Total Rows: {master_df.shape[0]:,}')
print(f'Total Columns: {master_df.shape[1]}')


Reading and combining 10 files...

✓ Loaded: ../data/raw/cftc/historicals/c_year 10.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 8.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 9.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 2.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 3.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 4.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 5.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 7.txt
✓ Loaded: ../data/raw/cftc/historicals/c_year 6.txt

🎉 Success! All historical text files have been combined.
Master Dataframe Shape: (119065, 191)
Total Rows: 119,065
Total Columns: 191


In [ ]:

# ============================================================
# 1. DIMENSION COLUMN MAPPINGS
# Actual column names in the source dataset.
# ============================================================

CFTC_DIMENSIONS = {
    "date": "report_date_as_yyyy-mm-dd",
    "as_of_date": "as_of_date_in_form_yymmdd",
    "market": "market_and_exchange_names",
    "contract_code": "cftc_contract_market_code",
    "market_code": "cftc_market_code",
    "commodity_code": "cftc_commodity_code",
    "region_code": "cftc_region_code",
    "report_type": "futonly_or_combined",
}


# ============================================================
# 2. DISPLAY LABELS
# Used for Streamlit filters, chart titles, legends and tables.
# ============================================================

CFTC_GROUP_LABELS = {
    "all": "All Contracts",
    "old": "Old Crop",
    "other": "Other Crop",
}

CFTC_TRADER_LABELS = {
    "prod_merc": "Producer / Merchant",
    "swap": "Swap Dealers",
    "m_money": "Managed Money",
    "other_rept": "Other Reportables",
    "tot_rept": "Total Reportables",
    "nonrept": "Non-reportables",
}

CFTC_POSITION_LABELS = {
    "long": "Long",
    "short": "Short",
    "spread": "Spread",
    "net": "Net",
}

CFTC_METRIC_LABELS = {
    "positions": "Positions",
    "net_positions": "Net Positions",
    "changes": "Change from Previous Report",
    "open_interest": "Open Interest",
    "open_interest_pct": "Open Interest (%)",
    "position_pct": "Percentage of Open Interest",
    "trader_counts": "Number of Traders",
    "concentration": "Position Concentration",
}

CFTC_CONCENTRATION_LABELS = {
    "gross_le_4": "Gross Positions — Top 4 Traders",
    "gross_le_8": "Gross Positions — Top 8 Traders",
    "net_le_4": "Net Positions — Top 4 Traders",
    "net_le_8": "Net Positions — Top 8 Traders",
}

CFTC_DIMENSION_LABELS = {
    "date": "Report Date",
    "as_of_date": "As-of Date",
    "market": "Market",
    "contract_code": "Contract Code",
    "market_code": "Market Code",
    "commodity_code": "Commodity Code",
    "region_code": "Region Code",
    "report_type": "Report Type",
}

CFTC_METADATA_LABELS = {
    "contract_units": "Contract Units",
    "subgroup_code": "CFTC Subgroup Code",
    "quote_contract_code": "Quote Contract Code",
    "quote_market_code": "Quote Market Code",
    "quote_commodity_code": "Quote Commodity Code",
}


# ============================================================
# 3. GROUPS, TRADERS AND POSITION TYPES
# Internal keys used to organise mappings.
# ============================================================

CFTC_GROUPS = tuple(CFTC_GROUP_LABELS)

CFTC_TRADERS = tuple(CFTC_TRADER_LABELS)

CFTC_POSITION_TYPES = {
    "long": "Long",
    "short": "Short",
    "spread": "Spread",
}

# Spread columns are available for these trader categories.
CFTC_TRADER_POSITION_TYPES = {
    trader: (
        ("long", "short", "spread")
        if trader in ("swap", "m_money", "other_rept")
        else ("long", "short")
    )
    for trader in CFTC_TRADERS
}


# ============================================================
# 4. REUSABLE POSITION COLUMN BUILDER
# ============================================================

def build_position_columns(pattern):
    """
    Build mappings in the following structure:

        mapping[group][trader][position_type] = column_name

    Example:
        CFTC_POSITIONS["all"]["m_money"]["long"]
        -> m_money_positions_long_all
    """

    return {
        group: {
            trader: {
                position_type: pattern.format(
                    trader=trader,
                    position_type=position_type,
                    group=group,
                )
                for position_type in CFTC_TRADER_POSITION_TYPES[trader]
            }
            for trader in CFTC_TRADERS
        }
        for group in CFTC_GROUPS
    }


# ============================================================
# 5. OPEN INTEREST
# ============================================================

CFTC_OPEN_INTEREST = {
    group: f"open_interest_{group}"
    for group in CFTC_GROUPS
}

CFTC_OPEN_INTEREST_PCT = {
    group: f"pct_of_open_interest_{group}"
    for group in CFTC_GROUPS
}


# ============================================================
# 6. LONG, SHORT AND SPREAD POSITIONS
# ============================================================

CFTC_POSITIONS = build_position_columns(
    "{trader}_positions_{position_type}_{group}"
)

CFTC_SPREAD_POSITIONS = {
    group: {
        trader: CFTC_POSITIONS[group][trader]["spread"]
        for trader in CFTC_TRADERS
        if "spread" in CFTC_TRADER_POSITION_TYPES[trader]
    }
    for group in CFTC_GROUPS
}


# ============================================================
# 7. CHANGES FROM PREVIOUS REPORT
# These source fields use the "all" grouping.
# ============================================================

CFTC_CHANGES_FROM_PREVIOUS_REPORT = {
    "open_interest": {
        "all": "change_in_open_interest_all",
    },
    "prod_merc": {
        "long": "change_in_prod_merc_long_all",
        "short": "change_in_prod_merc_short_all",
    },
    "swap": {
        "long": "change_in_swap_long_all",
        "short": "change_in_swap_short_all",
        "spread": "change_in_swap_spread_all",
    },
    "m_money": {
        "long": "change_in_m_money_long_all",
        "short": "change_in_m_money_short_all",
        "spread": "change_in_m_money_spread_all",
    },
    "other_rept": {
        "long": "change_in_other_rept_long_all",
        "short": "change_in_other_rept_short_all",
        "spread": "change_in_other_rept_spread_all",
    },
    "tot_rept": {
        "long": "change_in_tot_rept_long_all",
        "short": "change_in_tot_rept_short_all",
    },
    "nonrept": {
        "long": "change_in_nonrept_long_all",
        "short": "change_in_nonrept_short_all",
    },
}


# ============================================================
# 8. PERCENTAGE OF OPEN INTEREST
# ============================================================

CFTC_POSITION_PCT = build_position_columns(
    "pct_of_oi_{trader}_{position_type}_{group}"
)


# ============================================================
# 9. TRADER COUNTS
# ============================================================

CFTC_TRADER_COUNTS = {
    group: {
        trader: {
            position_type: (
                f"traders_{trader}_{position_type}_{group}"
            )
            for position_type in CFTC_TRADER_POSITION_TYPES[trader]
        }
        for trader in CFTC_TRADERS
    }
    for group in CFTC_GROUPS
}

CFTC_TRADER_COUNTS["all"]["total"] = "traders_tot_all"
CFTC_TRADER_COUNTS["old"]["total"] = "traders_tot_old"
CFTC_TRADER_COUNTS["other"]["total"] = "traders_tot_other"


# ============================================================
# 10. POSITION CONCENTRATION
# ============================================================

CFTC_CONCENTRATION_TYPES = {
    "gross_le_4": "Gross positions, top 4 traders",
    "gross_le_8": "Gross positions, top 8 traders",
    "net_le_4": "Net positions, top 4 traders",
    "net_le_8": "Net positions, top 8 traders",
}

CFTC_CONCENTRATION = {
    group: {
        concentration: {
            side: f"conc_{concentration}_tdr_{side}_{group}"
            for side in ("long", "short")
        }
        for concentration in CFTC_CONCENTRATION_TYPES
    }
    for group in CFTC_GROUPS
}


# ============================================================
# 11. OTHER METADATA COLUMN MAPPINGS
# ============================================================

CFTC_METADATA = {
    "contract_units": "contract_units",
    "subgroup_code": "cftc_subgroup_code",
    "quote_contract_code": "cftc_contract_market_code_quotes",
    "quote_market_code": "cftc_market_code_quotes",
    "quote_commodity_code": "cftc_commodity_code_quotes",
}


# ============================================================
# 12. HELPER FUNCTIONS
# ============================================================

def add_net_positions(
    df,
    traders=("m_money",),
    groups=("all",),
):
    """
    Add net-position columns.

    Net position = Long positions - Short positions.
    """

    df = df.copy()

    for group in groups:
        for trader in traders:
            long_col = CFTC_POSITIONS[group][trader]["long"]
            short_col = CFTC_POSITIONS[group][trader]["short"]

            df[long_col] = pd.to_numeric(
                df[long_col],
                errors="coerce",
            )

            df[short_col] = pd.to_numeric(
                df[short_col],
                errors="coerce",
            )

            df[f"{trader}_net_{group}"] = (
                df[long_col] - df[short_col]
            )

    return df


def validate_cftc_columns(df):
    """
    Check whether mapped source columns exist in a DataFrame.

    Returns a dictionary of missing columns by mapping category.
    """

    available = set(df.columns)
    missing = {}

    mappings = {
        "dimensions": CFTC_DIMENSIONS,
        "open_interest": CFTC_OPEN_INTEREST,
        "open_interest_pct": CFTC_OPEN_INTEREST_PCT,
        "metadata": CFTC_METADATA,
    }

    for category, mapping in mappings.items():
        absent = [
            column
            for column in mapping.values()
            if column not in available
        ]

        if absent:
            missing[category] = absent

    nested_mappings = {
        "positions": CFTC_POSITIONS,
        "position_pct": CFTC_POSITION_PCT,
        "trader_counts": CFTC_TRADER_COUNTS,
        "concentration": CFTC_CONCENTRATION,
    }

    for category, mapping in nested_mappings.items():
        absent = []

        def collect_columns(value):
            if isinstance(value, dict):
                for nested_value in value.values():
                    collect_columns(nested_value)
            elif isinstance(value, str) and value not in available:
                absent.append(value)

        collect_columns(mapping)

        if absent:
            missing[category] = sorted(set(absent))

    return missing


In [46]:
missing = validate_cftc_columns(master_df)
print(missing)

{'trader_counts': ['traders_nonrept_long_all', 'traders_nonrept_long_old', 'traders_nonrept_long_other', 'traders_nonrept_short_all', 'traders_nonrept_short_old', 'traders_nonrept_short_other']}
